# 03 — Weekly decision

The Friday-before-deadline routine. Run top to bottom:

1. refresh the snapshot (network, ~2–3 min with player history)
2. rebuild the warehouse
3. render the report to `reports/GWxx.md`
4. dig into any call you want to challenge

Commit the report afterwards — it's the decision log.

In [ ]:
# Run this cell first in every notebook.
# It makes the package importable from the repo root and loads settings from config/settings.toml.
# autoreload: edits under src/ take effect on the next cell run, no kernel restart needed.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from fpl_analysis.config import load_settings
from fpl_analysis.store import connect, query

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
settings = load_settings()
print("project root:", settings.project_root)
print("warehouse   :", settings.duckdb_path, "| exists:", settings.duckdb_path.exists())

In [ ]:
# 1-3: the same thing as `fpl run` from a terminal, then the report rendered inline.
from IPython.display import Markdown, display
from fpl_analysis.ingest import create_snapshot, latest_snapshot
from fpl_analysis.store import build
from fpl_analysis.report import build_report

REFRESH = True            # set False to re-use the latest snapshot on disk
INCLUDE_HISTORY = True    # False = fast run without the per-player endpoint

if REFRESH:
    snap = create_snapshot(settings, include_player_history=INCLUDE_HISTORY)
else:
    snap = latest_snapshot(settings)
print("snapshot:", snap.snapshot_id, "| current GW:", snap.current_gameweek, "| next GW:", snap.next_gameweek)

result = build(settings, snap)
print("models built:", len(result["models_built"]), "| history rows added:", result["history_rows_added"])
print("entry id:", settings.entry_id, "| squad override applied:", result["squad_override_applied"],
      "| chip squads XI EP:", result["chip_squads"])

ctx = build_report(settings, snap)
print("report:", ctx.path)
display(Markdown(ctx.markdown))   # rendered from THIS run — no separate display cell to go stale

## Challenge a call

The report ranks; you decide. The cells below pull the evidence behind the two decisions that matter most.

In [ ]:
# Captaincy: show the per-fixture breakdown for the top candidates you own
query(settings, """
SELECT e.web_name, e.fixture_label, e.form_used, e.season_ppg, e.xgi_used, e.base_rate,
       e.availability, e.fdr_multiplier, e.home_multiplier, e.expected_points
FROM marts.mart_player_expected_points AS e
JOIN marts.mart_squad AS s ON s.player_id = e.player_id
WHERE e.horizon_index = 1
ORDER BY e.expected_points DESC
""")

In [ ]:
# Transfers: for one outgoing player, the full candidate list with the constraints visible
OUT_PLAYER = None   # e.g. "Haaland"; None = the squad player with the lowest horizon EP

sql = """
SELECT out_web_name, out_ep_horizon, candidate_rank, in_web_name, in_team, in_price_m, in_selected_by_pct,
       in_form, in_xgi_per_90, in_fixture_run, in_ep_next, in_ep_horizon, ep_horizon_gain, bank_after_m
FROM marts.mart_transfer_candidates
WHERE out_web_name = COALESCE(?, (SELECT web_name FROM marts.mart_squad ORDER BY ep_horizon LIMIT 1))
ORDER BY candidate_rank
"""
query(settings, sql, [OUT_PLAYER])

## What-if: change the model weights without editing config

Settings can be overridden per process with environment variables. This re-runs only the marts under a heavier form weighting so you can see how much the ranking moves — a cheap sensitivity check.

In [ ]:
import os
from fpl_analysis.store import connect, run_models

os.environ.update({"FPL_ANALYSIS_WEIGHT_FORM": "0.6", "FPL_ANALYSIS_WEIGHT_SEASON_PPG": "0.2", "FPL_ANALYSIS_WEIGHT_XGI": "0.2"})
alt = load_settings()
con = connect(alt)
try:
    run_models(con, alt, only={"mart_player_form", "mart_player_expected_points", "mart_player_horizon"})
    moved = con.execute("""
        SELECT web_name, position_code, ep_horizon, rank_overall
        FROM marts.mart_player_horizon ORDER BY ep_horizon DESC LIMIT 15
    """).df()
finally:
    con.close()
for k in ("FPL_ANALYSIS_WEIGHT_FORM", "FPL_ANALYSIS_WEIGHT_SEASON_PPG", "FPL_ANALYSIS_WEIGHT_XGI"):
    os.environ.pop(k, None)
moved
# Re-run `fpl build` afterwards to restore the configured weights in the warehouse.